# LeKiwi MobileNet + MiniTransformer Training

**Structured training notebook for LeKiwi robot policy learning**

- **Vision**: MobileNetV3-Small (ImageNet pretrained)
- **Policy**: MiniTransformer (2+2 layers)
- **Target**: 9 DOF action space (6 arm + 3 base)
- **Data**: LeRobot dataset format
- **Hardware**: M4 MacBook local training

## Setup & Imports

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import json

# LeRobot imports
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

/Users/yj/Workspaces/lerobot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.7.1
Device: mps


## Model Architecture: MobileNet + MiniTransformer

In [ ]:
class MobileNetMiniTransformer(nn.Module):
    """
    LeKiwi Policy: MobileNet vision + MiniTransformer temporal reasoning
    
    Input: Image [B, 3, 224, 224] + State [B, 9]
    Output: Actions [B, chunk_size, 9]
    """
    
    def __init__(self, 
                 chunk_size=20,
                 state_dim=9,
                 action_dim=9,
                 hidden_dim=256,
                 num_encoder_layers=2,
                 num_decoder_layers=2,
                 num_heads=4):
        super().__init__()
        
        self.chunk_size = chunk_size
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        
        # Vision backbone: MobileNetV3-Small (ImageNet pretrained)
        self.vision_backbone = torchvision.models.mobilenet_v3_small(pretrained=True)
        # Remove classifier, use as feature extractor
        self.vision_backbone.classifier = nn.Identity()
        vision_features = 576  # MobileNetV3-Small output
        
        # Project vision features to hidden dim
        self.vision_proj = nn.Linear(vision_features, hidden_dim)
        
        # Project robot state to hidden dim  
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        
        # MiniTransformer encoder (combines vision + state)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        
        # Learnable action queries for decoder
        self.action_queries = nn.Parameter(torch.randn(chunk_size, hidden_dim))
        
        # MiniTransformer decoder (generates action sequence)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)
        
        # Action head: project to motor coordinates
        self.action_head = nn.Linear(hidden_dim, action_dim)
        
        print(f"Model created: {self.count_parameters():,} parameters")
        
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
        
    def forward(self, image, state):
        """
        Forward pass: image + state → action sequence
        
        Args:
            image: [B, 3, 224, 224] Camera input
            state: [B, 9] Current robot state (6 arm + 3 base)
            
        Returns:
            actions: [B, chunk_size, 9] Predicted action sequence
        """
        batch_size = image.shape[0]
        
        # Vision encoding
        vision_features = self.vision_backbone(image)  # [B, 576]
        vision_embed = self.vision_proj(vision_features)  # [B, 256]
        vision_embed = vision_embed.unsqueeze(1)  # [B, 1, 256]
        
        # State encoding
        state_embed = self.state_proj(state)  # [B, 256]
        state_embed = state_embed.unsqueeze(1)  # [B, 1, 256]
        
        # Combine vision + state for encoder
        encoder_input = torch.cat([state_embed, vision_embed], dim=1)  # [B, 2, 256]
        
        # Encoder: contextual understanding
        memory = self.encoder(encoder_input)  # [B, 2, 256]
        
        # Decoder: generate action sequence
        # Expand action queries for batch
        queries = self.action_queries.unsqueeze(0).expand(batch_size, -1, -1)  # [B, chunk_size, 256]
        
        # Decode actions
        action_features = self.decoder(queries, memory)  # [B, chunk_size, 256]
        
        # Project to action space
        actions = self.action_head(action_features)  # [B, chunk_size, 9]
        
        return actions

# Test model creation
model = MobileNetMiniTransformer()
print(f"Model parameters: {model.count_parameters():,}")

# Test forward pass
test_image = torch.randn(2, 3, 224, 224)
test_state = torch.randn(2, 9)
test_output = model(test_image, test_state)
print(f"Test output shape: {test_output.shape}")

## Dataset Loading & Preparation

In [ ]:
# Available LeRobot datasets (choose one to start)
available_datasets = [
    "lerobot/aloha_sim_insertion_human",  # ALOHA simulation
    "lerobot/pusht",                     # 2D pushing task
    "lerobot/xarm_pick_medium",          # XArm manipulation
]

print("Available datasets:")
for i, dataset in enumerate(available_datasets):
    print(f"{i+1}. {dataset}")

# For MVP, start with PushT (simpler 2D task)
dataset_name = "lerobot/pusht"
print(f"\nUsing dataset: {dataset_name}")

In [ ]:
class LeKiwiDataset(torch.utils.data.Dataset):
    """
    Wrapper for LeRobot datasets to work with our MobileNet model
    
    Handles:
    - Image preprocessing (resize to 224x224)
    - State/action dimension mapping
    - Normalization
    """
    
    def __init__(self, dataset_name, chunk_size=20, train=True):
        self.chunk_size = chunk_size
        
        # Load LeRobot dataset
        try:
            self.lerobot_dataset = LeRobotDataset(dataset_name)
            print(f"Loaded dataset: {dataset_name}")
            print(f"Total episodes: {len(self.lerobot_dataset.episode_data_index)}")
            print(f"Total frames: {len(self.lerobot_dataset)}")
        except Exception as e:
            print(f"Error loading dataset {dataset_name}: {e}")
            print("Using dummy data for development...")
            self.use_dummy = True
            return
            
        self.use_dummy = False
        
        # Get dataset info
        sample = self.lerobot_dataset[0]
        print("Sample keys:", list(sample.keys()))
        
        # Find image and state keys
        self.image_keys = [k for k in sample.keys() if 'image' in k.lower()]
        self.state_keys = [k for k in sample.keys() if 'state' in k.lower()]
        self.action_keys = [k for k in sample.keys() if 'action' in k.lower()]
        
        print(f"Image keys: {self.image_keys}")
        print(f"State keys: {self.state_keys}")
        print(f"Action keys: {self.action_keys}")
        
        # Use first available keys
        self.image_key = self.image_keys[0] if self.image_keys else None
        self.state_key = self.state_keys[0] if self.state_keys else None
        self.action_key = self.action_keys[0] if self.action_keys else None
        
        if sample[self.image_key] is not None:
            print(f"Image shape: {sample[self.image_key].shape}")
        if sample[self.state_key] is not None:
            print(f"State shape: {sample[self.state_key].shape}")
        if sample[self.action_key] is not None:
            print(f"Action shape: {sample[self.action_key].shape}")
            
        # Image preprocessing
        self.image_transform = torchvision.transforms.Compose([
            torchvision.transforms.ToPILImage(),
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                std=[0.229, 0.224, 0.225]
            )
        ])
        
    def __len__(self):
        if self.use_dummy:
            return 1000  # Dummy dataset size
        return len(self.lerobot_dataset)
    
    def __getitem__(self, idx):
        if self.use_dummy:
            # Generate dummy data for development
            image = torch.randn(3, 224, 224)
            state = torch.randn(9)  # 9 DOF state
            action = torch.randn(9)  # 9 DOF action
            return {
                'image': image,
                'state': state,
                'action': action
            }
        
        # Get sample from LeRobot dataset
        sample = self.lerobot_dataset[idx]
        
        # Process image
        image = sample[self.image_key]
        if image is not None:
            # Convert from tensor to numpy if needed
            if isinstance(image, torch.Tensor):
                image = image.numpy()
            
            # Handle different image formats
            if image.dtype == np.uint8:
                image = image.astype(np.float32) / 255.0
            
            # Ensure correct channel order (H, W, C)
            if image.shape[0] == 3:  # If channels first
                image = np.transpose(image, (1, 2, 0))
            
            # Apply transforms
            image = self.image_transform(image)
        else:
            image = torch.zeros(3, 224, 224)
        
        # Process state
        state = sample[self.state_key]
        if state is not None:
            if isinstance(state, np.ndarray):
                state = torch.from_numpy(state).float()
            # Pad or truncate to 9 dimensions for LeKiwi
            if len(state) < 9:
                state = torch.cat([state, torch.zeros(9 - len(state))])
            elif len(state) > 9:
                state = state[:9]
        else:
            state = torch.zeros(9)
        
        # Process action
        action = sample[self.action_key]
        if action is not None:
            if isinstance(action, np.ndarray):
                action = torch.from_numpy(action).float()
            # Pad or truncate to 9 dimensions for LeKiwi
            if len(action) < 9:
                action = torch.cat([action, torch.zeros(9 - len(action))])
            elif len(action) > 9:
                action = action[:9]
        else:
            action = torch.zeros(9)
        
        return {
            'image': image,
            'state': state,
            'action': action
        }

# Create dataset
try:
    train_dataset = LeKiwiDataset(dataset_name, chunk_size=20)
    print(f"Dataset created successfully!")
except Exception as e:
    print(f"Dataset creation failed: {e}")
    print("Using dummy dataset for development")
    train_dataset = LeKiwiDataset("dummy", chunk_size=20)

# Test dataset
sample = train_dataset[0]
print(f"Sample image shape: {sample['image'].shape}")
print(f"Sample state shape: {sample['state'].shape}")
print(f"Sample action shape: {sample['action'].shape}")

## Training Setup

In [ ]:
class LeKiwiTrainer:
    """
    Trainer for LeKiwi MobileNet policy
    
    Features:
    - Action chunking (predict sequence of actions)
    - MSE loss for continuous action prediction
    - Learning rate scheduling
    - Progress tracking
    """
    
    def __init__(self, model, device, 
                 learning_rate=1e-4,
                 backbone_lr_ratio=0.1,
                 weight_decay=1e-4):
        self.model = model.to(device)
        self.device = device
        
        # Different learning rates for backbone vs policy
        backbone_params = []
        policy_params = []
        
        for name, param in model.named_parameters():
            if 'vision_backbone' in name:
                backbone_params.append(param)
            else:
                policy_params.append(param)
        
        # Optimizer with different learning rates
        self.optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': learning_rate * backbone_lr_ratio},
            {'params': policy_params, 'lr': learning_rate}
        ], weight_decay=weight_decay)
        
        # Learning rate scheduler
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=1000, eta_min=learning_rate * 0.01
        )
        
        # Loss function
        self.criterion = nn.MSELoss()
        
        # Training history
        self.history = {
            'train_loss': [],
            'learning_rates': []
        }
        
        print(f"Trainer initialized:")
        print(f"  Backbone params: {len(backbone_params):,}")
        print(f"  Policy params: {len(policy_params):,}")
        print(f"  Backbone LR: {learning_rate * backbone_lr_ratio:.2e}")
        print(f"  Policy LR: {learning_rate:.2e}")
    
    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        
        pbar = tqdm(dataloader, desc="Training")
        for batch in pbar:
            # Move batch to device
            images = batch['image'].to(self.device)
            states = batch['state'].to(self.device)
            actions = batch['action'].to(self.device)
            
            # Forward pass
            predicted_actions = self.model(images, states)  # [B, chunk_size, 9]
            
            # For now, use only first predicted action vs target action
            # TODO: Implement proper action chunking with future actions
            pred_action = predicted_actions[:, 0, :]  # [B, 9]
            target_action = actions  # [B, 9]
            
            # Compute loss
            loss = self.criterion(pred_action, target_action)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            
            # Update metrics
            total_loss += loss.item()
            num_batches += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg_loss': f'{total_loss/num_batches:.4f}'
            })
        
        # Step scheduler
        self.scheduler.step()
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
        current_lr = self.optimizer.param_groups[1]['lr']  # Policy LR
        
        return avg_loss, current_lr
    
    def train(self, dataloader, epochs=100, save_every=25):
        print(f"Starting training for {epochs} epochs...")
        print(f"Batch size: {dataloader.batch_size}")
        print(f"Batches per epoch: {len(dataloader)}")
        
        start_time = time.time()
        best_loss = float('inf')
        
        for epoch in range(epochs):
            epoch_start = time.time()
            
            # Train epoch
            avg_loss, current_lr = self.train_epoch(dataloader)
            
            # Record history
            self.history['train_loss'].append(avg_loss)
            self.history['learning_rates'].append(current_lr)
            
            # Calculate timing
            epoch_time = time.time() - epoch_start
            total_time = time.time() - start_time
            
            # Print progress
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {avg_loss:.6f} | "
                  f"LR: {current_lr:.2e} | "
                  f"Time: {epoch_time:.1f}s | "
                  f"Total: {total_time/60:.1f}m")
            
            # Save best model
            if avg_loss < best_loss:
                best_loss = avg_loss
                self.save_model('best_model.pth')
                print(f"  💾 Saved best model (loss: {best_loss:.6f})")
            
            # Periodic saves
            if (epoch + 1) % save_every == 0:
                self.save_model(f'checkpoint_epoch_{epoch+1}.pth')
                print(f"  💾 Saved checkpoint")
        
        total_time = time.time() - start_time
        print(f"\n🎉 Training completed in {total_time/60:.1f} minutes")
        print(f"Best loss: {best_loss:.6f}")
        
        return self.history
    
    def save_model(self, filename):
        """Save model checkpoint"""
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'history': self.history
        }
        
        # Create models directory
        models_dir = Path('../models')
        models_dir.mkdir(exist_ok=True)
        
        torch.save(checkpoint, models_dir / filename)
    
    def load_model(self, filename):
        """Load model checkpoint"""
        checkpoint = torch.load(Path('../models') / filename, map_location=self.device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.history = checkpoint['history']
        
        print(f"Model loaded from {filename}")

# Create trainer
model = MobileNetMiniTransformer().to(device)
trainer = LeKiwiTrainer(model, device, learning_rate=1e-4)

print("Trainer ready!")

## Training Execution

In [ ]:
# Create data loader
batch_size = 32 if device.type == 'mps' else 16  # Adjust for M4 MacBook
dataloader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=0,  # Set to 0 for MPS compatibility
    pin_memory=False
)

print(f"DataLoader created:")
print(f"  Batch size: {batch_size}")
print(f"  Batches per epoch: {len(dataloader)}")
print(f"  Total samples: {len(train_dataset)}")

# Test batch
test_batch = next(iter(dataloader))
print(f"\nTest batch shapes:")
print(f"  Images: {test_batch['image'].shape}")
print(f"  States: {test_batch['state'].shape}")
print(f"  Actions: {test_batch['action'].shape}")

In [ ]:
# Start training
print("🚀 Starting LeKiwi MobileNet training...")
print(f"Device: {device}")
print(f"Model parameters: {model.count_parameters():,}")

# Train for different durations based on setup
if train_dataset.use_dummy:
    epochs = 10  # Quick test with dummy data
    print("⚠️  Using dummy data - running short test")
else:
    epochs = 100  # Full training with real data
    print(f"📊 Using real dataset - training for {epochs} epochs")

# Execute training
history = trainer.train(dataloader, epochs=epochs, save_every=25)

print("\n✅ Training completed!")

## Training Visualization & Results

In [ ]:
# Plot training results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(history['train_loss'])
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.grid(True)

# Learning rate curve
ax2.plot(history['learning_rates'])
ax2.set_title('Learning Rate Schedule')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_yscale('log')
ax2.grid(True)

plt.tight_layout()
plt.show()

# Print final statistics
print(f"\n📈 Training Summary:")
print(f"  Final loss: {history['train_loss'][-1]:.6f}")
print(f"  Best loss: {min(history['train_loss']):.6f}")
print(f"  Total epochs: {len(history['train_loss'])}")
print(f"  Final LR: {history['learning_rates'][-1]:.2e}")

## Model Testing & Validation

In [ ]:
# Test trained model
model.eval()
with torch.no_grad():
    test_batch = next(iter(dataloader))
    images = test_batch['image'].to(device)
    states = test_batch['state'].to(device)
    target_actions = test_batch['action'].to(device)
    
    # Forward pass
    predicted_actions = model(images, states)
    
    # Get first action from sequence
    pred_action = predicted_actions[:, 0, :]  # [B, 9]
    
    # Compute test loss
    test_loss = F.mse_loss(pred_action, target_actions)
    
    print(f"\n🧪 Model Testing:")
    print(f"  Test loss: {test_loss.item():.6f}")
    print(f"  Input shapes: {images.shape}, {states.shape}")
    print(f"  Output shape: {predicted_actions.shape}")
    
    # Show example predictions
    print(f"\n📊 Example Predictions (first sample):")
    print(f"  Target action:    {target_actions[0].cpu().numpy()}")
    print(f"  Predicted action: {pred_action[0].cpu().numpy()}")
    print(f"  Absolute error:   {torch.abs(pred_action[0] - target_actions[0]).cpu().numpy()}")
    
    # Action statistics
    mean_error = torch.mean(torch.abs(pred_action - target_actions), dim=0)
    print(f"\n📏 Mean Absolute Error per DOF:")
    for i, error in enumerate(mean_error):
        dof_name = f"DOF_{i+1:02d}" if i < 6 else f"Base_{i-5}"
        print(f"  {dof_name}: {error.item():.4f}")

## Model Export for Deployment

In [ ]:
# Export model for deployment
def export_model_for_deployment():
    """Export trained model for Pi 5 deployment"""
    
    # Load best model
    try:
        trainer.load_model('best_model.pth')
        print("✅ Loaded best model")
    except:
        print("⚠️  Using current model (best model not found)")
    
    model.eval()
    
    # Create deployment directory
    deploy_dir = Path('../deployment')
    deploy_dir.mkdir(exist_ok=True)
    
    # Save model state dict only (for faster loading)
    torch.save(model.state_dict(), deploy_dir / 'lekiwi_mobilenet_model.pth')
    
    # Save model configuration
    config = {
        'model_type': 'MobileNetMiniTransformer',
        'chunk_size': model.chunk_size,
        'state_dim': model.state_dim,
        'action_dim': model.action_dim,
        'hidden_dim': model.hidden_dim,
        'input_image_size': [224, 224],
        'normalization_mean': [0.485, 0.456, 0.406],
        'normalization_std': [0.229, 0.224, 0.225],
        'total_parameters': model.count_parameters(),
        'training_loss': min(history['train_loss']) if history['train_loss'] else None
    }
    
    with open(deploy_dir / 'model_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    # Test inference speed
    test_image = torch.randn(1, 3, 224, 224).to(device)
    test_state = torch.randn(1, 9).to(device)
    
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(test_image, test_state)
    
    # Time inference
    num_runs = 100
    start_time = time.time()
    
    with torch.no_grad():
        for _ in range(num_runs):
            output = model(test_image, test_state)
    
    inference_time = (time.time() - start_time) / num_runs * 1000  # ms
    
    print(f"\n🚀 Model Export Summary:")
    print(f"  Model saved to: {deploy_dir / 'lekiwi_mobilenet_model.pth'}")
    print(f"  Config saved to: {deploy_dir / 'model_config.json'}")
    print(f"  Model parameters: {config['total_parameters']:,}")
    print(f"  Inference time (M4): {inference_time:.1f}ms")
    print(f"  Expected Pi 5 time: ~{inference_time * 2:.1f}ms")
    print(f"  Target frequency: ~{1000/(inference_time*2):.0f}Hz")
    
    return deploy_dir

# Export the model
deploy_path = export_model_for_deployment()
print(f"\n✅ Model ready for deployment!")
print(f"📁 Files in {deploy_path}:")
for file in deploy_path.glob('*'):
    size = file.stat().st_size / (1024*1024)  # MB
    print(f"  {file.name}: {size:.1f}MB")

## Next Steps

1. **Model trained and exported** ✅
2. **Deploy to Pi 5**: Transfer model files to Raspberry Pi
3. **Create inference script**: Real-time camera → action pipeline
4. **Test with LeKiwi robot**: Validate motor commands
5. **Collect real data**: Record LeKiwi demonstrations
6. **Retrain**: Improve with robot-specific data

**Performance Targets:**
- Inference: ~60ms on Pi 5 (16Hz control)
- Accuracy: 70-80% task success after real data training
- Memory: <1GB RAM usage

**Files for deployment:**
- `lekiwi_mobilenet_model.pth`: Trained model weights
- `model_config.json`: Model configuration
- Next: Create `lekiwi_deploy.py` inference script